# PeerConf vs DeepConf — same harness, one switch

Runs Mofe's PeerConf and a DeepConf-style fixed warm-up **through the same code**,
so the only thing that differs between the two runs is when the cutoff is decided.

**Read this before you start.** The model is 8B, about 16 GB in fp16.

| Where | Works? |
|---|---|
| Colab free (1× T4, 16 GB) | **No** — weights alone fill the card |
| Kaggle (2× T4, 32 GB) | **Smoke test only** — this notebook, shrunken settings |
| AWS `ml.g6e.xlarge` (L40S, 48 GB) | **Yes** — the real runs go here |

On 2× T4 there is roughly 14 GB left after the weights, which is not enough for 16
concurrent traces of 30k tokens. So this notebook deliberately shrinks the race to
prove the switch works end to end. **Numbers from this notebook are not results** —
they tell you the plumbing is correct, nothing more.

## 1. Check the GPUs

On Kaggle pick **GPU T4 × 2** in Settings → Accelerator.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch; print("GPUs visible:", torch.cuda.device_count())


## 2. Install and fetch the code

In [ ]:
!pip install vllm 2>&1 | tail -20
!which vllm || echo "no vllm binary on PATH - the -m form below handles that"
!python -c "import vllm; print('vllm', vllm.__version__)"
!git clone -q --branch feat/deepconf-warmup-baseline https://github.com/yityler/Algoverse-AI-Research.git || echo "already cloned"
%cd Algoverse-AI-Research/peerconf
!ls


## 3. Start the model server

Two changes from `cell1_start_server.py`: split across both T4s, and a much smaller
context so the KV cache fits. First run downloads ~16 GB, so give it time.

In [ ]:
import subprocess, time, requests, os
MODEL  = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"
SERVER = "http://localhost:8000"
os.makedirs("peerconf_out", exist_ok=True)

proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", "8000",
     "--tensor-parallel-size", "2",     # split the 8B across both T4s
     "--dtype", "half",                 # T4 is Turing: no bfloat16
     "--max-model-len", "4096",         # shrunk to fit; raise on a 48GB card
     "--gpu-memory-utilization", "0.92",
     "--max-logprobs", "20"],
    stdout=open("vllm_server.log","w"), stderr=subprocess.STDOUT)

t0=time.time()
while True:
    try:
        if requests.get(f"{SERVER}/health", timeout=5).status_code==200:
            print(f"server up after {time.time()-t0:.0f}s"); break
    except requests.exceptions.RequestException: pass
    if proc.poll() is not None: raise RuntimeError("server died - check vllm_server.log")
    if time.time()-t0 > 2400: raise RuntimeError("no server after 40 min - check vllm_server.log")
    time.sleep(10)


In [ ]:
# if the server dies, this is where it says why
!tail -25 vllm_server.log


## 4. Shrink the race

`cell2_race.py` keeps its settings as constants at the top, so this rewrites them into a
copy rather than editing the original. Nothing here touches the committed file.

Note **WARMUP_TRACES must be smaller than MAX_TRACES** — otherwise the warm-up never
finishes and DeepConf mode silently never judges anything. The script now refuses to run
in that state, but it is the thing to double-check when you scale these back up.

In [ ]:
import re

SMOKE = {
    "QIDS":          "range(2)",   # 2 questions, not 30
    "SEATS":         "6",          # 6 concurrent, not 16
    "MAX_TRACES":    "10",         # launch cap
    "WARMUP_TRACES": "5",          # must be < MAX_TRACES
    "MAX_TOK_TRACE": "3000",       # not 30000
    "WINDOW":        "512",        # must be << MAX_TOK_TRACE or no window ever fills
    "DWELL_TOKENS":  "64",
    "FINAL_CHECK_TOKENS": "800",
}

def build(mode_on, path):
    src = open("cell2_race.py").read()
    for k, v in SMOKE.items():
        src = re.sub(rf"^{k}(\s*)=\s*\S+", f"{k}\\1= {v}", src, count=1, flags=re.M)
    src = re.sub(r"^WARMUP_MODE(\s*)=\s*\S+", f"WARMUP_MODE\\1= {mode_on}", src,
                 count=1, flags=re.M)
    open(path, "w").write(src)
    # echo back what actually landed, so a silent regex miss can't fool us
    got = {k: re.search(rf"^{k}\s*=\s*(\S+)", src, re.M).group(1)
           for k in list(SMOKE) + ["WARMUP_MODE"]}
    print(path, "->", got)

build("False", "run_peerconf.py")
build("True",  "run_deepconf.py")


## 5. Run PeerConf (line redrawn live)

In [ ]:
!python run_peerconf.py 2>&1 | tail -40


## 6. Run DeepConf (line frozen after warm-up)

Watch for the `[warm-up] complete ... line FROZEN` line — that is the switch working.

In [ ]:
!python run_deepconf.py 2>&1 | tail -40


## 7. Compare

Tokens is the number that matters. Accuracy on 2 questions is meaningless — it is here
only to confirm nothing catastrophically broke.

In [ ]:
import pickle, glob, os, re
from collections import defaultdict

rows = defaultdict(lambda: {"tokens": 0, "correct": 0, "n": 0, "warm": None})
for p in sorted(glob.glob("peerconf_out/*.pkl")):
    d = pickle.load(open(p, "rb"))
    mode = "deepconf" if d["config"].get("WARMUP_MODE") else "peerconf"
    ans = d["voting"]["majority"][0]
    gt  = d["gt"]
    r = rows[mode]
    r["tokens"] += d["tokens"]; r["n"] += 1
    r["correct"] += int(str(ans).strip() == str(gt).strip())
    if d["config"].get("frozen_line") is not None:
        r["warm"] = d["config"]["frozen_line"]

print(f"{'mode':<12}{'questions':<11}{'tokens':<12}{'tok/question':<14}{'majority correct'}")
print("-"*62)
for m, r in rows.items():
    if not r["n"]: continue
    print(f"{m:<12}{r['n']:<11}{r['tokens']:<12}{r['tokens']//r['n']:<14}{r['correct']}/{r['n']}")

if len(rows) == 2:
    p, d = rows["peerconf"], rows["deepconf"]
    if d["tokens"]:
        saving = 100*(d["tokens"]-p["tokens"])/d["tokens"]
        print(f"\nPeerConf uses {saving:+.1f}% tokens vs DeepConf on this (tiny) sample.")
    print("\n2 questions proves the plumbing, not the claim. The real run is 30 questions")
    print("on a 48GB card, and even then report tokens as the headline, not accuracy.")


## 8. When you move to the real hardware

On an `ml.g6e.xlarge` (L40S, 48 GB) drop the `--tensor-parallel-size` flag, set
`--dtype bfloat16`, raise `--max-model-len` back to 32768, and skip the shrink cell
entirely so the original settings apply.

Then, for the threshold sweep Mofe asked for: hold `WARMUP_MODE = False` and vary
`LINE_TOP` over 0.80 / 0.90 / 0.95 / 0.97. Filenames now carry the mode and threshold,
so runs will not overwrite each other.